In [157]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pickle
from copy import copy
from math import pi, sqrt
%matplotlib qt

parent_dir = os.path.abspath("..")
if parent_dir not in sys.path:
    print(f"appending {parent_dir} to path")
    sys.path.append(parent_dir)

import src.unit_conv as conv


quantum_root = Path(parent_dir).parents[0]
fname = "./70x70x30small/all/time_evol_runs.pickle"
fname = quantum_root / fname

with open(fname, 'rb') as f:
    runs = pickle.load(f)

xr = [runs[0]["r_right"][0][0]]
xl = [runs[0]["r_left"][0][0]]
xrm = [runs[0]["r_right_max"][0][0]]
xlm = [runs[0]["r_left_max"][0][0]]

times = [0]
n_part = runs[0]["n_part"]

e_tot = [runs[0]["energies"]["total"][0]*conv.hartree_to_nK]
e_kin = [runs[0]["energies"]["kinetic"][0]*conv.hartree_to_nK]
e_ext = [runs[0]["energies"]["ext"][0]*conv.hartree_to_nK]
e_contact = [runs[0]["energies"]["contact"][0]*conv.hartree_to_nK]
e_contact_ext = [runs[0]["energies"]["contact_ext"][0]*conv.hartree_to_nK]
e_dd = [runs[0]["energies"]["dipole"][0]*conv.hartree_to_nK]

for i, run in enumerate(runs):
    r_right = run["r_right"][0][1:]
    r_left = run["r_left"][0][1:]
    rm_right = run["r_right_max"][0][1:]
    rm_left = run["r_left_max"][0][1:]

    tot = run["energies"]["total"][1:]*conv.hartree_to_nK
    kin = run["energies"]["kinetic"][1:]*conv.hartree_to_nK
    ext = run["energies"]["ext"][1:]*conv.hartree_to_nK
    contact = run["energies"]["contact"][1:]*conv.hartree_to_nK
    contact_ext = run["energies"]["contact_ext"][1:]*conv.hartree_to_nK
    dd = run["energies"]["dipole"][1:]*conv.hartree_to_nK

    n_steps = run["n_steps"]
    dt = run["timestep"]*conv.jat_to_ms
    timestamps = np.arange(1, n_steps + 1)*dt
    times.extend(timestamps + copy(times[-1]))

    xr.extend(r_right.copy())
    xl.extend(r_left.copy())
    xrm.extend(rm_right.copy())
    xlm.extend(rm_left.copy())

    e_tot.extend(tot.copy())
    e_kin.extend(kin.copy())
    e_ext.extend(ext.copy())
    e_contact.extend(contact.copy())
    e_contact_ext.extend(contact_ext.copy())
    e_dd.extend(dd.copy())

xr = np.array(xr)
xl = np.array(xl)
xrm = np.array(xrm)
xlm = np.array(xlm)

e_tot = np.array(e_tot)
e_kin = np.array(e_kin)
e_ext = np.array(e_ext)
e_contact = np.array(e_contact)
e_contact_ext = np.array(e_contact_ext)
e_dd = np.array(e_dd)

times = np.array(times)

## Classical Kinetic and classical dipole-dipole energy ##

In [158]:
h_eff = 1.88
m = 163.929*1822.888353
dist = xr - xl
v_rel = np.empty_like(dist)
v_rel[0] = (dist[1] - dist[0])/(times[1] - times[0])
v_rel[-1] = (dist[-1] - dist[-2])/(times[-1] - times[-2])

p1 = slice(2, None, 1)
m1 = slice(0, -2, 1)
zero = slice(1, -1, 1)
v_rel[zero] = (dist[p1] - dist[m1])/2/(times[p1] - times[zero])

E_k = 0.25*m*v_rel**2*(conv.mum_to_atm*conv.jat_to_ms)**2*conv.hartree_to_nK


def dd_classic(d, h):
    add = 131
    m = 163.929*1822.888353
    rho = (n_part/2/h)
    cdd = 12*pi*add/m*conv.hartree_to_nK/conv.mum_to_atm**3
    return cdd/4/pi*rho**2/d*2*(1 - 1/sqrt((h/d)**2 + 1))

vdd_classic = np.empty_like(dist)
for i in range(dist.shape[0]):
    vdd_classic[i] = dd_classic(dist[i], h_eff)

## Difference between both of the approximations and the "true quantities" ##

In [159]:
dVdd = (e_dd - e_dd[0])/n_part - (vdd_classic - vdd_classic[0])/n_part
dkin = (e_kin - e_kin[0])/n_part - E_k - E_k[0]
ext = (e_ext - e_ext[0])/n_part

fig, ax = plt.subplots()
ax.plot(times, dVdd, label='Vdd: quantum - classical')
ax.plot(times, -dkin, label='Kin: -quantum + classical')
ax.plot(times, ext, label=r"$\Delta$ external")
ax.plot(times, dVdd + dkin, label="dVdd + dEkin")
plt.legend()

In [160]:
fig, ax = plt.subplots()

ax.plot(times, dVdd, label="Vdd:")
ax.plot(times, dVdd + dkin, label="dVdd + dEkin")

In [161]:
fig, ax = plt.subplots()

ax.plot(times, (vdd_classic - vdd_classic[0])/n_part + (e_kin - e_kin[0])/n_part)
ax.plot(times, E_k)

## MISC ##

In [162]:
fig, ax = plt.subplots(1,2)
ax = ax.ravel()

ax[0].plot(times, (e_kin - e_kin[0])/n_part, label="Wartość dla rachunku kwantowego")
ax[0].plot(times, E_k - E_k[0], label="Wartość klasyczna")
ax[0].legend()
ax[0].set(xlabel=r"t[ms]", ylabel=r"E [nK]")

ax[1].plot(dist, (e_kin - e_kin[0])/n_part, "r.", label="wartość kwantowa")
ax[1].plot(dist, E_k - E_k[0], "k.", label="Wartość klasyczna")
ax[1].legend()
ax[1].set(xlabel=r"d[$\mu$ m]", ylabel=r"E [nK]")


[Text(0.5, 0, 'd[$\\mu$ m]'), Text(0, 0.5, 'E [nK]')]

In [163]:
fig, ax = plt.subplots(1,2)
ax = ax.ravel()

ax[0].plot(times, (e_dd - e_dd[0])/n_part, label="Wartość dla rachunku kwantowego")
ax[0].plot(times, (vdd_classic - vdd_classic[0])/n_part, label="Wartość uzyskana dla dwóch patyków")
ax[0].legend()
ax[0].set(xlabel=r"t[ms]", ylabel=r"E [nK]")

ax[1].plot(dist, (e_dd - e_dd[0])/n_part, 'r.' ,label="Wartość dla rachunku kwantowego")
ax[1].plot(dist, (vdd_classic - vdd_classic[0])/n_part, 'k.' ,label="Wartość uzyskana dla dwóch patyków")
ax[1].legend()
ax[1].set(xlabel=r"d[$\mu$ m]", ylabel=r"E [nK]")

[Text(0.5, 0, 'd[$\\mu$ m]'), Text(0, 0.5, 'E [nK]')]

In [164]:
fig, ax = plt.subplots(1,2)
ax = ax.ravel()

ax[0].plot(times, (e_dd - e_dd[0] + e_kin - e_kin[0])/n_part, label="Wartość dla rachunku kwantowego")
ax[0].plot(times, (vdd_classic - vdd_classic[0])/n_part + E_k - E_k[0], label="Wartość uzyskana dla dwóch patyków")
ax[0].legend()
ax[0].set(xlabel=r"t[ms]", ylabel=r"E [nK]")

ax[1].plot(dist, (e_dd - e_dd[0])/n_part, 'r.' ,label="Wartość dla rachunku kwantowego")
ax[1].plot(dist, (vdd_classic - vdd_classic[0])/n_part, 'k.' ,label="Wartość uzyskana dla dwóch patyków")
ax[1].legend()
ax[1].set(xlabel=r"d[$\mu$ m]", ylabel=r"E [nK]")

[Text(0.5, 0, 'd[$\\mu$ m]'), Text(0, 0.5, 'E [nK]')]

In [165]:
fig, ax = plt.subplots()

ax.plot(times, (e_kin - e_kin[0])/n_part - E_k + E_k[0])
ax.plot(times, (e_dd - e_dd[0] - vdd_classic + vdd_classic[0])/n_part)

fig, ax = plt.subplots(1,2)
ax = ax.ravel()

ax[0].plot(times, (e_kin - e_kin[0])/n_part - E_k + E_k[0] + (e_dd - e_dd[0] - vdd_classic + vdd_classic[0])/n_part)
ax[1].plot(dist, (e_kin - e_kin[0])/n_part - E_k + E_k[0] + (e_dd - e_dd[0] - vdd_classic + vdd_classic[0])/n_part)

In [166]:
fig, ax = plt.subplots(1, 2)
ax = ax.ravel()

ax[0].plot(times, (e_kin - e_kin[0])/n_part + (e_dd - e_dd[0])/n_part - (e_ext - e_ext[0])/n_part)
ax[1].plot(dist, (e_kin - e_kin[0])/n_part + (e_dd - e_dd[0])/n_part - - (e_ext - e_ext[0])/n_part)

## CZEGO MI BRAKUJE DO SZCZĘŚCIA ##

In [169]:
fig, ax = plt.subplots(1, 2)
ax = ax.ravel()

ax[0].plot(times, -(E_k - E_k[0]) - (vdd_classic - vdd_classic[0])/n_part - (e_ext - e_ext[0])/n_part, label="Brakujący potencjał")
ax[0].set(xlabel="t [ms]", ylabel="E [nk]")
ax[0].legend()
ax[1].plot(dist, -(E_k - E_k[0]) - (vdd_classic - vdd_classic[0])/n_part - (e_ext - e_ext[0])/n_part, 'k.', label="Brakujący potencjał")
ax[1].set(xlabel=r"d [$\mu$ m]", ylabel="E [nk]")
ax[1].legend()

/home/artur/inzyn/quantum_decaf/view/.venv/lib/python3.14/site-packages/matplotlib/cbook.py:1811: ComplexWarning: Casting complex values to real discards the imaginary part
  return math.isfinite(val)
/home/artur/inzyn/quantum_decaf/view/.venv/lib/python3.14/site-packages/matplotlib/cbook.py:1408: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asanyarray(x, float)
